# 01 — R0~R3 smoke

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

각 objective를 1,200/300행, 최대 40 optimizer step으로 실행합니다. smoke 결과는 우승자 선정에 사용하지 않습니다. T4 런타임을 사용하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive = '/content/drive/MyDrive/LLM_Router/colab-inputs/routing-objective-colab-code.tar.gz'
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install 'transformers==4.48.3' 'safetensors>=0.4'
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil, subprocess
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요.'


In [ ]:
ROOT = pathlib.Path('/content/drive/MyDrive/LLM_Router/routing-objective-comparison/v1')
DATA = ROOT / 'dataset'
if not (DATA / '_SUCCESS').is_file(): raise RuntimeError('00 notebook을 먼저 완료하세요.')
LOCAL = pathlib.Path('/content/objective-data')
if LOCAL.exists(): shutil.rmtree(LOCAL)
shutil.copytree(DATA, LOCAL)


In [ ]:
for method in ('ordinal', 'pointwise', 'pairwise', 'regret'):
    out = pathlib.Path('/content/smoke') / method / 'seed-42'
    command = [
        'routing-objective-comparison', '--config', 'ml/config/routing_objective_comparison_v1.json',
        'train', '--method', method, '--training-path', str(LOCAL/'train.parquet'),
        '--validation-path', str(LOCAL/'validation.parquet'), '--outcome-manifest-path', str(LOCAL/'manifest.json'),
        '--output-dir', str(out), '--seed', '42', '--maximum-train-rows', '1200',
        '--maximum-validation-rows', '300', '--maximum-train-steps', '40'
    ]
    subprocess.run(command, check=True)
    assert (out / '_SUCCESS').is_file()
print('네 smoke 완료')


In [ ]:
target = ROOT / 'smoke'
staging = ROOT / 'smoke.staging'
if staging.exists(): shutil.rmtree(staging)
shutil.copytree('/content/smoke', staging)
if target.exists(): raise RuntimeError('같은 smoke version이 이미 있습니다. 새 version을 사용하세요.')
os.replace(staging, target)
print('게시 완료:', target)
